<a href="https://colab.research.google.com/github/msolowes-star/DI-Bootcamp/blob/main/Week_16_Mini_Project_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Mini-Project 2: Scraping “Scrape This Site” - Frames Page

Install and Import the libraries

In [6]:

!pip -q install -U selenium beautifulsoup4 pandas

!which google-chrome
!which chromium
!which chromedriver


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.0 which is incompatible.
/usr/bin/chromedriver


In [8]:

!pip -q install pandas==2.2.3


In [1]:

!apt-get update -qq
!apt-get install -y chromium chromium-driver > /dev/null 2>&1

!which chromium
!which chromedriver

!chromium --version
!chromedriver --version


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
/usr/bin/chromedriver
/bin/bash: line 1: chromium: command not found

Command '/usr/bin/chromedriver' requires the chromium snap to be installed.
Please install it with:

snap install chromium



In [2]:

import pandas as pd
import time

from bs4 import BeautifulSoup
from urllib.parse import urljoin

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import chromedriver_autoinstaller


Initialize Selenium WebDriver

In [3]:
import os
import subprocess
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
import chromedriver_autoinstaller

# 1. Download and install standard Google Chrome (stable edition)
print("Downloading and installing stable Google Chrome...")
subprocess.run("wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -", shell=True, check=True)
subprocess.run("echo 'deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main' >> /etc/apt/sources.list.d/google-chrome.list", shell=True, check=True)
subprocess.run("apt-get update -qq && apt-get install -y google-chrome-stable", shell=True, check=True)

# 2. Use chromedriver_autoinstaller to automatically download matching ChromeDriver
print("Configuring ChromeDriver...")
driver_path = chromedriver_autoinstaller.install()

# 3. Configure Selenium Options
chrome_options = Options()
chrome_options.add_argument("--headless=new")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")
chrome_options.add_argument("--window-size=1920,1080")

# 4. Initialize WebDriver
driver = webdriver.Chrome(
    service=Service(driver_path),
    options=chrome_options
)

print("Selenium WebDriver initialized successfully!")
print("Chrome version:", driver.capabilities["browserVersion"])

Configuring ChromeDriver...
Selenium WebDriver initialized successfully!
Chrome version: 157.0.8103.38


Navigate to the Frames page

In [4]:

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

url = "https://www.scrapethissite.com/pages/frames/"

driver.get(url)

wait = WebDriverWait(driver, 30)

wait.until(
    EC.presence_of_element_located(
        (By.CSS_SELECTOR, "iframe, frame")
    )
)

print("Page Title:", driver.title)
print("Current URL:", driver.current_url)
print("Page loaded successfully!")


Page Title: Turtles All the Way Down: Frames & iFrames | Scrape This Site | A public sandbox for learning web scraping
Current URL: https://www.scrapethissite.com/pages/frames/
Page loaded successfully!


Identify the frames

In [5]:

frames = driver.find_elements(
    By.CSS_SELECTOR,
    "iframe, frame"
)

print("Number of frames found:", len(frames))

for i, frame in enumerate(frames):
    print(f"\nFrame {i + 1}")
    print("ID:", frame.get_attribute("id"))
    print("Name:", frame.get_attribute("name"))
    print("Source:", frame.get_attribute("src"))


Number of frames found: 1

Frame 1
ID: iframe
Name: 
Source: https://www.scrapethissite.com/pages/frames/?frame=i


Switch into the first frame

In [6]:

from bs4 import BeautifulSoup

driver.switch_to.default_content()

driver.switch_to.frame(
    driver.find_element(By.CSS_SELECTOR, "iframe, frame")
)

wait.until(
    EC.presence_of_element_located((By.TAG_NAME, "body"))
)

frame_soup = BeautifulSoup(
    driver.page_source,
    "html.parser"
)

print("FRAME CONTENT")
print("=" * 50)

print(frame_soup.get_text(" ", strip=True)[:1500])

print("\nHeadings:")
for heading in frame_soup.find_all(["h1", "h2", "h3", "h4"]):
    print(heading.get_text(" ", strip=True))


FRAME CONTENT
Turtles All the Way Down: Frames & iFrames | Scrape This Site | A public sandbox for learning web scraping Scrape This Site Sandbox Lessons FAQ Login Carettochelyidae Learn more → Cheloniidae Learn more → Chelydridae Learn more → Dermatemydidae Learn more → Dermochelyidae Learn more → Emydidae Learn more → Geoemydidae Learn more → Kinosternidae Learn more → Platysternidae Learn more → Testudinidae Learn more → Trionychidae Learn more → Chelidae Learn more → Pelomedusidae Learn more → Podocnemididae Learn more → Lessons and Videos © Hartley Brody 2023

Headings:
Carettochelyidae
Cheloniidae
Chelydridae
Dermatemydidae
Dermochelyidae
Emydidae
Geoemydidae
Kinosternidae
Platysternidae
Testudinidae
Trionychidae
Chelidae
Pelomedusidae
Podocnemididae


Extract turtle families and links

In [7]:

from urllib.parse import urljoin
import pandas as pd

turtle_data = []

# Find the headings containing turtle family names
headings = frame_soup.find_all(["h3", "h4"])

for heading in headings:
    family_name = heading.get_text(" ", strip=True)

    # Find the corresponding "Learn more" link
    container = heading.parent
    link = container.find("a", href=True)

    # If not found, check the next sibling
    if not link:
        link = heading.find_next("a", href=True)

    family_url = (
        urljoin(
            "https://www.scrapethissite.com",
            link["href"]
        )
        if link else "Not available"
    )

    turtle_data.append({
        "Family": family_name,
        "Details URL": family_url
    })

df_turtles = pd.DataFrame(turtle_data)

print("Turtle families extracted:", len(df_turtles))
display(df_turtles)


Turtle families extracted: 14


,Family,Details URL
0,Carettochelyidae,https://www.scrapethissite.com/pages/frames/?f...
1,Cheloniidae,https://www.scrapethissite.com/pages/frames/?f...
2,Chelydridae,https://www.scrapethissite.com/pages/frames/?f...
3,Dermatemydidae,https://www.scrapethissite.com/pages/frames/?f...
4,Dermochelyidae,https://www.scrapethissite.com/pages/frames/?f...
5,Emydidae,https://www.scrapethissite.com/pages/frames/?f...
6,Geoemydidae,https://www.scrapethissite.com/pages/frames/?f...
7,Kinosternidae,https://www.scrapethissite.com/pages/frames/?f...
8,Platysternidae,https://www.scrapethissite.com/pages/frames/?f...
9,Testudinidae,https://www.scrapethissite.com/pages/frames/?f...


Save the data to CSV

In [8]:

df_turtles.to_csv(
    "turtle_families.csv",
    index=False,
    encoding="utf-8-sig"
)

print("CSV saved successfully!")
print("File: turtle_families.csv")


CSV saved successfully!
File: turtle_families.csv


Close Selenium

In [9]:

driver.switch_to.default_content()
driver.quit()

print("Selenium WebDriver closed successfully!")


Selenium WebDriver closed successfully!


### Mini-Project 2: Scraping Data from Frames

**Objective:** Extract information from an iframe using Selenium and BeautifulSoup.

**Methodology:**

1. Initialized Selenium WebDriver with headless Chrome.
2. Navigated to the Scrape This Site Frames page.
3. Identified the iframe containing turtle family information.
4. Used Selenium's `switch_to.frame()` method to access the iframe.
5. Parsed the iframe's HTML using BeautifulSoup.
6. Extracted turtle family names and their corresponding detail-page links.
7. Organized the extracted information into a pandas DataFrame.
8. Exported the dataset to `turtle_families.csv`.
9. Closed the Selenium WebDriver session.

**Result:** Successfully extracted information about 14 turtle families from an HTML iframe.

**Key skills practiced:** Selenium browser automation, iframe navigation, HTML parsing, BeautifulSoup, pandas, and CSV export.